# Admissions Review Workflow: Missed Opportunity and Documentation Support
This notebook applies the trained CAR024 model to the MIMIC proof-of-concept dataset. It demonstrates how model probability scores can be converted into two operational review queues:

1. **Potential Missed Opportunity Review**<br>
   _CAR024 is absent, but the model predicts it is likely._

2. **Documentation Support Review**<br>
   _CAR024 is present, but the model assigns a low probability, suggesting the procedure may warrant documentation or coding support review._

The model does not determine whether billing errors, omissions, or inappropriate billing occurred. <br>
It prioritizes admissions for human review. <br>
The outputs are intended for dashboard development and operational review prioritization.

In [1]:
import pandas as pd
import joblib
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### Data directory setup
This notebook assumes that you can access the **Quantic Capstone** Google Drive set up for this project as a local path on your personal computer.<br>
Before running the notebook, set `base_directory` in the cell immediately below to the folder location on your machine.

If your local path is different, update `base_directory` so it points to your own accessible **Quantic Capstone** directory before running the next cell.


In [2]:
base_directory = "/Users/scott/Library/CloudStorage/GoogleDrive-clt.av8or@gmail.com/My Drive/Quantic Capstone"

## Configuration and Inputs

This section defines the local project directory, the MIMIC proof-of-concept input file, the saved CAR024 model artifact, and the target procedure indicator. The saved model artifact contains the fitted model, selected threshold, and feature list required for scoring.

In [3]:
# Configuration
working_dir= base_directory + '/Proof of Concept/'
input_file_nm='mimic_input_car024.csv'
model_path= base_directory + '/Models/diagnosis_procedure_model/flaml_car024_model.pkl'
target_col='target_prccsr_car024'
target_ccsr_val = "CAR024"

In [4]:
# Import admission-level dataset
df = pd.read_csv(working_dir + input_file_nm)

In [5]:
# Load saved CAR024 model artifact
model_artifacts=joblib.load(model_path)

In [6]:
# # Extract fitted model, selected threshold, and expected feature list
model=model_artifacts["model"]
threshold=model_artifacts['threshold']
features=model_artifacts['features']
target=model_artifacts['target']

In [7]:
# Confirm Model Components
print("Target:",target)
print("Threshold:",threshold)
print("Number of Features:", len(features))

Target: target_prccsr_car024
Threshold: 0.23
Number of Features: 31


In [8]:
# Confirm that all features expected by the saved model exist in the MIMIC input dataset.
# Expected result: empty list.
missing_cols = [
    col for col in features
    if col not in df.columns
]

print(missing_cols)

[]


In [9]:
# Score admissions
X_mimic = df[features].copy()

df["y_prob"] = (
    model.predict_proba(X_mimic)[:, 1]
)

df["y_pred"] = (
    df["y_prob"] >= threshold
).astype(int)

In [10]:
# Model sanity checks
print("Admissions:",
      len(df))

print("Actual prevalence:",
      f"{df[target_col].mean():.4%}")

print("Predicted prevalence:",
      f"{df['y_pred'].mean():.4%}")

df["y_prob"].describe()

Admissions: 27107
Actual prevalence: 7.2417%
Predicted prevalence: 14.7121%


count    27107.000000
mean         0.107662
std          0.163209
min          0.000998
25%          0.013761
50%          0.037187
75%          0.119013
max          0.930530
Name: y_prob, dtype: float64

## Scoring Sanity Check Interpretation

The probability distribution remains well behaved when the HCUP-trained CAR024 model is applied to the MIMIC proof-of-concept dataset. <br>
This suggests the model generalizes reasonably to an independent clinical dataset despite differences in patient population, coding practices, and source system structure.

In [11]:
# Evaluate external model performance on the MIMIC proof-of-concept dataset.
# These metrics describe how well the HCUP-trained CAR024 model generalizes to MIMIC.
y_true = df[target_col]
y_pred = df["y_pred"]
y_prob = df["y_prob"]

precision=precision_score(y_true, y_pred)
recall=recall_score(y_true, y_pred)
f1=f1_score(y_true, y_pred)
roc_auc=roc_auc_score(y_true, y_prob)

print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1: {f1}")
print(f"ROC-AUC: {roc_auc}")

Precision: 0.35155466399197594
Recall: 0.7142129393785023
F1: 0.471181314064863
ROC-AUC: 0.9069691840409329


In [12]:
# Lift
actual_prevalence = df[target_col].mean()
lift = precision / actual_prevalence

print(f"Actual Prevalence: {actual_prevalence:.4%}")
print(f"Precision: {precision:.4%}")
print(f"Lift: {lift:.2f}x")

Actual Prevalence: 7.2417%
Precision: 35.1555%
Lift: 4.85x


Interpretation:<br>
The model concentrates likely CAR024 encounters into a subset nearly five times denser than random review.

The model shows strong external performance on the MIMIC proof-of-concept dataset, supporting its use as a review-prioritization workflow.

## Review Candidate Generation

After confirming model performance on the MIMIC dataset, the next step is to convert probability scores into operational review candidates.<br>
This section creates review queues that could be used by revenue integrity teams to prioritize admissions for further examination.

## Threshold Review Summary Function

The function below creates threshold comparison tables for both review workflows.<br>
It allows the same logic to be reused for missed-opportunity cases and documentation-support cases, improving consistency across dashboard outputs.

In [13]:
# Build Threshold Comparison Table
# Use of AI: ChatGPT was used to develop this function.
def build_threshold_review_summary(
    df,
    scenarios,
    target_col,
    workflow_type,
    delta_col="delta_charge"
):
    """
    Build threshold comparison table for either:
    - missed_opportunity: target absent, predicted likely
    - documentation_support: target present, predicted unlikely
    """

    rows = []

    for scenario in scenarios:
        scenario_name = scenario["scenario"]
        t = scenario["threshold"]

        temp_df = df.copy()

        if workflow_type == "missed_opportunity":
            temp_df["review_case"] = (
                (temp_df[target_col] == 0)
                &
                (temp_df["y_prob"] >= t)
            ).astype(int)

            review_cases = temp_df[temp_df["review_case"] == 1].copy()

            review_cases["potential_financial_impact"] = (
                review_cases["y_prob"]
                *
                review_cases[delta_col]
            )

        elif workflow_type == "documentation_support":
            temp_df["review_case"] = (
                (temp_df[target_col] == 1)
                &
                (temp_df["y_prob"] <= t)
            ).astype(int)

            review_cases = temp_df[temp_df["review_case"] == 1].copy()

            review_cases["support_risk_weight"] = (
                1 - review_cases["y_prob"]
            )

            review_cases["potential_financial_impact"] = (
                review_cases["support_risk_weight"]
                *
                review_cases[delta_col]
            )

        else:
            raise ValueError(
                "workflow_type must be 'missed_opportunity' or 'documentation_support'"
            )

        rows.append({
            "Workflow": workflow_type,
            "Scenario": scenario_name,
            "Threshold": t,
            "Review Cases": review_cases.shape[0],
            "Review Rate": temp_df["review_case"].mean(),
            "Median Case Impact": review_cases["potential_financial_impact"].median(),
            "Average Case Impact": review_cases["potential_financial_impact"].mean(),
            "Total Potential Impact": review_cases["potential_financial_impact"].sum()
        })

    return pd.DataFrame(rows)

## Workflow 1: Potential Missed Opportunity Review
CAR024 is absent, but the model predicts it is likely. These cases may represent potential missed billing opportunities if manual review confirms the procedure was clinically performed and appropriate for billing.

In [14]:
df["missed_opportunity_case"] = (
    (df[target_col] == 0)
    &
    (df["y_pred"] == 1)
).astype(int)

missed_opportunity_df = df[
    df["missed_opportunity_case"] == 1
].copy()

In [15]:
# Summarize the missed-opportunity review queue at the selected operating threshold.
print("Missed Opportunity Cases:", f"{len(missed_opportunity_df):,}")
print("Review Rate:", f"{len(missed_opportunity_df) / len(df):.4%}")
print("Average Probability:", f"{missed_opportunity_df['y_prob'].mean():.4f}")
print("Median Probability:", f"{missed_opportunity_df['y_prob'].median():.4f}")

Missed Opportunity Cases: 2,586
Review Rate: 9.5400%
Average Probability: 0.3922
Median Probability: 0.3623


This summary describes the size of the missed-opportunity review queue at the selected operating threshold and the model’s average confidence among flagged cases.


In [16]:
# Rank Highest-Confidence Candidates
missed_opportunity_df = missed_opportunity_df.sort_values(
    by="y_prob",
    ascending=False
).reset_index(drop=True)

In [17]:
# Create a clean review table
missed_opportunity_sample = missed_opportunity_df[
    [
        "admit_id",
        "drg_cd",
        "y_prob",
        "age",
        "gender_female",
        "rom_score",
        "soi_score",
        "length_of_stay",
        "diagnosis_count",
        "distinct_ccsr_count",
        "secondary_ccsr_count"
    ]
].head(25)

missed_opportunity_sample.style.format({
    "y_prob": "{:.3f}",
    "length_of_stay": "{:.1f}",
    "age": "{:.0f}",
    "rom_score": "{:.0f}",
    "soi_score": "{:.0f}",
    "diagnosis_count": "{:.0f}",
    "distinct_ccsr_count": "{:.0f}",
    "secondary_ccsr_count": "{:.0f}"
})

,admit_id,drg_cd,y_prob,age,gender_female,rom_score,soi_score,length_of_stay,diagnosis_count,distinct_ccsr_count,secondary_ccsr_count
0,29285346,710,0.895,41,1,4,4,66.0,52,32,50
1,20289704,710,0.887,55,1,4,4,48.0,45,34,43
2,20493673,710,0.885,47,1,4,4,35.0,36,31,34
3,29876293,710,0.871,65,1,4,4,46.0,48,36,46
4,29657639,710,0.864,40,0,4,4,24.0,58,36,56
5,29551743,710,0.858,67,0,4,4,28.0,42,30,40
6,27177542,710,0.854,62,1,4,4,24.0,39,28,37
7,20156661,710,0.853,54,0,4,4,45.0,47,35,45
8,24120519,710,0.851,63,1,4,4,23.0,38,32,36
9,26033399,710,0.850,34,0,4,4,50.0,42,35,40


In [18]:
# Profile Review Candidates by DRG
review_drg_summary = (
    missed_opportunity_df
    .groupby("drg_cd")
    .agg(
        review_cases=("admit_id", "count"),
        avg_probability=("y_prob", "mean"),
        median_probability=("y_prob", "median")
    )
    .reset_index()
    .sort_values(
        by="review_cases",
        ascending=False
    )
)

review_drg_summary

,drg_cd,review_cases,avg_probability,median_probability
4,710,1311,0.429400,0.395136
1,194,739,0.350340,0.322492
3,469,280,0.394612,0.367245
0,139,149,0.326948,0.296773
2,383,107,0.311245,0.279082


### Top 20 Highest Probability Cases for Review

In [19]:
top_candidates = missed_opportunity_df[
    [
        "admit_id",
        "drg_cd",
        "y_prob",
        "age",
        "gender_female",
        "rom_score",
        "soi_score",
        "length_of_stay",
        "diagnosis_count",
        "distinct_ccsr_count",
        "secondary_ccsr_count"
    ]
].head(20)

top_candidates.style.format({
    "y_prob": "{:.3f}",
    "age": "{:.0f}",
    "length_of_stay": "{:.1f}",
    "rom_score": "{:.0f}",
    "soi_score": "{:.0f}",
    "diagnosis_count": "{:.0f}",
    "distinct_ccsr_count": "{:.0f}",
    "secondary_ccsr_count": "{:.0f}"
})

,admit_id,drg_cd,y_prob,age,gender_female,rom_score,soi_score,length_of_stay,diagnosis_count,distinct_ccsr_count,secondary_ccsr_count
0,29285346,710,0.895,41,1,4,4,66.0,52,32,50
1,20289704,710,0.887,55,1,4,4,48.0,45,34,43
2,20493673,710,0.885,47,1,4,4,35.0,36,31,34
3,29876293,710,0.871,65,1,4,4,46.0,48,36,46
4,29657639,710,0.864,40,0,4,4,24.0,58,36,56
5,29551743,710,0.858,67,0,4,4,28.0,42,30,40
6,27177542,710,0.854,62,1,4,4,24.0,39,28,37
7,20156661,710,0.853,54,0,4,4,45.0,47,35,45
8,24120519,710,0.851,63,1,4,4,23.0,38,32,36
9,26033399,710,0.850,34,0,4,4,50.0,42,35,40


Interpretation:<br>
The highest-confidence review candidates exhibited consistent clinical characteristics, including maximum severity scores, high diagnosis burden, and prolonged hospital stays. These patterns suggest the model is identifying clinically complex encounters where the target procedure could plausibly be expected rather than randomly flagging admissions.

## Potential Financial Opportunity
_This represents an estimate of potential opportunity if manual review confirms the procedure was clinically performed and appropriate for billing._<br><br>
The model is currently optimized at a threshold of 0.23; we’ll include 4 other thresholds, 0.15, 0.19, 0.27, 0.31.<br>
The threshold acts like a review-intensity dial.<br>
RuraHealth can adjust it depending on staffing, audit goals, or tolerance for false positives.<br>
| Strategy | Threshold |
| :--- | :---: |
|Very Aggressive | 0.15
|Aggressive | 0.19
|**Optimized**	| **0.23**
|Conservative | 0.27
|Very Conservative | 0.31

In [20]:
# Add CAR024 median DRG-level charge deltas from HCUP/NIS benchmark analysis.
# These represent benchmark-based potential billing lift by DRG.
car024_delta_by_drg = {
    "139": 52420.5,
    "194": 59986.0,
    "383": 26995.0,
    "469": 61283.0,
    "710": 98751.0
}

df["delta_charge"] = df["drg_cd"].astype(str).map(car024_delta_by_drg)

# Sanity check: should be 0 if all rows are in the five modeled DRGs
print("Rows missing delta_charge:", df["delta_charge"].isna().sum())

Rows missing delta_charge: 0


In [21]:
# Define review thresholds
# 0.23 is the validation-selected operating threshold for CAR024
missed_opportunity_thresholds = [
    {"scenario": "Very Aggressive", "threshold": 0.15},
    {"scenario": "Aggressive", "threshold": 0.19},
    {"scenario": "Selected Operating Threshold", "threshold": 0.23},
    {"scenario": "Conservative", "threshold": 0.27},
    {"scenario": "Very Conservative", "threshold": 0.31}
]

### Five-Threshold Financial Comparison

This table shows how the size and value of the review queue changes as the model threshold is adjusted. Lower thresholds create a larger, more aggressive review queue, while higher thresholds create a smaller and more conservative queue. Expected opportunity is estimated as predicted probability multiplied by the DRG-level benchmark charge delta.

In [22]:
# Five-Threshold Financial Comparison Table

# Build missed-opportunity threshold summary using reusable function
missed_opportunity_summary = build_threshold_review_summary(
    df=df,
    scenarios=missed_opportunity_thresholds,
    target_col=target_col,
    workflow_type="missed_opportunity"
)

missed_opportunity_summary.style.format({
    "Threshold": "{:.2f}",
    "Review Cases": "{:,.0f}",
    "Review Rate": "{:.2%}",
    "Median Case Impact": "${:,.2f}",
    "Average Case Impact": "${:,.2f}",
    "Total Potential Impact": "${:,.2f}"
})

,Workflow,Scenario,Threshold,Review Cases,Review Rate,Median Case Impact,Average Case Impact,Total Potential Impact
0,missed_opportunity,Very Aggressive,0.15,"4,054",14.96%,"$19,185.80","$24,161.70","$97,951,531.45"
1,missed_opportunity,Aggressive,0.19,"3,177",11.72%,"$23,915.74","$28,103.61","$89,285,167.44"
2,missed_opportunity,Selected Operating Threshold,0.23,"2,586",9.54%,"$28,039.36","$31,456.16","$81,345,639.27"
3,missed_opportunity,Conservative,0.27,"2,108",7.78%,"$32,590.44","$34,683.77","$73,113,387.96"
4,missed_opportunity,Very Conservative,0.31,"1,711",6.31%,"$36,179.67","$37,990.38","$65,001,533.52"


## Workflow 2: Documentation Support Review

The first workflow identifies cases where CAR024 is absent but predicted likely. <br>
This complementary workflow looks in the opposite direction: CAR024 is present, but the model assigns a relatively low probability based on the admission’s diagnosis pattern and encounter characteristics.

These cases should not be interpreted as inappropriate billing or fraud. Rather, they may warrant documentation review to confirm that the procedure is adequately supported in the record.

In [23]:
documentation_support_thresholds = [
    {"scenario": "Very Strict", "threshold": 0.05},
    {"scenario": "Strict", "threshold": 0.10},
    {"scenario": "Moderate", "threshold": 0.15},
    {"scenario": "Broad", "threshold": 0.19},
    {"scenario": "Broadest", "threshold": 0.23}
]

documentation_support_summary = build_threshold_review_summary(
    df=df,
    scenarios=documentation_support_thresholds,
    target_col=target_col,
    workflow_type="documentation_support"
)

documentation_support_summary.style.format({
    "Threshold": "{:.2f}",
    "Review Cases": "{:,.0f}",
    "Review Rate": "{:.2%}",
    "Median Case Impact": "${:,.2f}",
    "Average Case Impact": "${:,.2f}",
    "Total Potential Impact": "${:,.2f}"
})

,Workflow,Scenario,Threshold,Review Cases,Review Rate,Median Case Impact,Average Case Impact,Total Potential Impact
0,documentation_support,Very Strict,0.05,104,0.38%,"$57,479.16","$53,845.57","$5,599,939.44"
1,documentation_support,Strict,0.10,238,0.88%,"$55,741.41","$53,208.53","$12,663,629.23"
2,documentation_support,Moderate,0.15,371,1.37%,"$54,204.29","$52,743.22","$19,567,734.10"
3,documentation_support,Broad,0.19,467,1.72%,"$53,028.66","$53,216.77","$24,852,233.09"
4,documentation_support,Broadest,0.23,561,2.07%,"$52,212.14","$53,222.65","$29,857,904.12"


In [24]:
## Workflow 2: Documentation Support Review
# CAR024 is present, but the model assigns a relatively low probability. These cases may warrant documentation or coding support review to confirm that the procedure is adequately supported in the record.

support_threshold = 0.23

df["documentation_support_case"] = (
    (df[target_col] == 1)
    &
    (df["y_prob"] <= support_threshold)
).astype(int)

documentation_support_df = df[
    df["documentation_support_case"] == 1
].copy()

documentation_support_df["support_risk_weight"] = (
    1 - documentation_support_df["y_prob"]
)

# Keep this for reference, but the most important value in this workflow
# is Support Risk Weight, calculated above
documentation_support_df["potential_financial_impact"] = (
    documentation_support_df["support_risk_weight"]
    *
    documentation_support_df["delta_charge"]
)

# documentation_support_df = documentation_support_df.sort_values(
#    by="potential_financial_impact",
#    ascending=False
#).reset_index(drop=True)

# Order the review queue by Support Risk Weight (desc)
documentation_support_df = documentation_support_df.sort_values(
    by="potential_financial_impact",
    ascending=False
).reset_index(drop=True)

In [25]:
support_review_sample = documentation_support_df[
    [
        "admit_id",
        "drg_cd",
        "y_prob",
        "support_risk_weight",
        "delta_charge",
        "potential_financial_impact",
        "age",
        "gender_female",
        "rom_score",
        "soi_score",
        "length_of_stay",
        "diagnosis_count",
        "distinct_ccsr_count",
        "secondary_ccsr_count"
    ]
].head(25)

support_review_sample.style.format({
    "y_prob": "{:.3f}",
    "support_risk_weight": "{:.3f}",
    "delta_charge": "${:,.2f}",
    "potential_financial_impact": "${:,.2f}",
    "age": "{:.0f}",
    "length_of_stay": "{:.1f}",
    "rom_score": "{:.0f}",
    "soi_score": "{:.0f}",
    "diagnosis_count": "{:.0f}",
    "distinct_ccsr_count": "{:.0f}",
    "secondary_ccsr_count": "{:.0f}"
})

,admit_id,drg_cd,y_prob,support_risk_weight,delta_charge,potential_financial_impact,age,gender_female,rom_score,soi_score,length_of_stay,diagnosis_count,distinct_ccsr_count,secondary_ccsr_count
0,27635276,710,0.034,0.966,"$98,751.00","$95,411.07",53,0,2,3,2.0,43,28,41
1,26234062,710,0.036,0.964,"$98,751.00","$95,171.66",88,1,2,2,4.0,22,18,20
2,29952243,710,0.037,0.963,"$98,751.00","$95,141.75",85,0,2,2,4.0,21,19,19
3,20970059,710,0.044,0.956,"$98,751.00","$94,363.94",91,1,3,3,4.0,13,9,11
4,25057168,710,0.045,0.955,"$98,751.00","$94,312.75",23,1,1,3,4.0,12,11,10
5,27715604,710,0.047,0.953,"$98,751.00","$94,134.18",67,0,2,3,4.0,10,9,8
6,24602094,710,0.047,0.953,"$98,751.00","$94,127.75",44,0,1,1,5.0,8,8,6
7,20622065,710,0.048,0.952,"$98,751.00","$93,982.92",39,0,2,3,4.0,12,12,10
8,29395169,710,0.067,0.933,"$98,751.00","$92,133.14",77,1,3,3,4.0,18,17,16
9,28902545,710,0.069,0.931,"$98,751.00","$91,897.77",60,1,4,3,3.0,14,13,12


In [26]:
documentation_support_drg_summary = (
    documentation_support_df
    .groupby("drg_cd")
    .agg(
        review_cases=("admit_id", "count"),
        avg_probability=("y_prob", "mean"),
        median_probability=("y_prob", "median"),
        median_case_impact=("potential_financial_impact", "median"),
        avg_case_impact=("potential_financial_impact", "mean"),
        total_potential_impact=("potential_financial_impact", "sum")
    )
    .reset_index()
    .sort_values(
        by="total_potential_impact",
        ascending=False
    )
)

documentation_support_drg_summary.style.format({
    "review_cases": "{:,.0f}",
    "avg_probability": "{:.3f}",
    "median_probability": "{:.3f}",
    "median_case_impact": "${:,.2f}",
    "avg_case_impact": "${:,.2f}",
    "total_potential_impact": "${:,.2f}"
})

,drg_cd,review_cases,avg_probability,median_probability,median_case_impact,avg_case_impact,total_potential_impact
1,194,302,0.114,0.108,"$53,519.36","$53,142.02","$16,048,890.23"
4,710,86,0.152,0.163,"$82,663.54","$83,727.45","$7,200,560.58"
0,139,61,0.095,0.084,"$48,036.29","$47,424.43","$2,892,890.19"
2,383,77,0.104,0.103,"$24,208.30","$24,188.31","$1,862,499.70"
3,469,35,0.136,0.146,"$52,331.10","$52,944.67","$1,853,063.41"


## Summary of Workflow Outputs

The notebook produces two complementary review queues:
* The missed-opportunity workflow supports potential under-coding review by identifying admissions where CAR024 is absent but predicted likely.
* The documentation-support workflow supports compliance-oriented review by identifying admissions where CAR024 is present despite relatively weak model support.

Together, these outputs demonstrate how the same predictive model can support both revenue opportunity identification and proactive documentation review.

## Dashboard Exports
Creates the four dashboard-ready files:<br>
* missed-opportunity threshold summary
* documentation-support threshold summary
* missed-opportunity queue
* documentation-support queue

In [27]:
# Dashboard Exports
missed_opportunity_summary.to_csv(
    working_dir + "financial_review_threshold_summary.csv",
    index=False
)

documentation_support_summary.to_csv(
    working_dir + "documentation_review_threshold_summary.csv",
    index=False
)

missed_opportunity_df.to_csv(
    working_dir + "financial_review_queue.csv",
    index=False
)

documentation_support_df.to_csv(
    working_dir + "documentation_review_queue.csv",
    index=False
)